# Query Gaia eDR3 Mock Catalog

The Gaia eDR3 Mock Catalog from [Rybizki et al. (2020)](https://iopscience.iop.org/article/10.1088/1538-3873/ab8cb0) is used to get an estimate on the number of stars with some attributes within 100 pc. This can be constructed by doing the below queries in the Gaia eDR3 Mock catalog.
```
SELECT 
    floor(source_id / ((power(2, 35)) * (power(4, (12 - 3))))) AS healpix_,
    floor((phot_g_mean_mag_ext - 3) / 0.25) AS phot_g_mean_mag_,
    floor(((phot_g_mean_mag_ext - phot_rp_mean_mag_ext) - (-0.4)) / 0.1) AS g_rp_,
    COUNT(*) AS n
FROM (
    SELECT GAVO_NORMAL_RANDOM(phot_g_mean_mag, phot_g_mean_mag_error) + a0*A0_1_gaia_g as phot_g_mean_mag_ext,
           GAVO_NORMAL_RANDOM(phot_rp_mean_mag, phot_rp_mean_mag_error) + a0*A0_1_gaia_rp as phot_rp_mean_mag_ext,
           source_id
    FROM gedr3mock.main
    JOIN gedr3mock.parsec_props
    USING (index_parsec)) as mock
WHERE phot_g_mean_mag_ext - phot_rp_mean_mag_ext > -0.4
      AND phot_g_mean_mag_ext - phot_rp_mean_mag_ext < 2.2
      AND phot_g_mean_mag_ext > 3 
      AND phot_g_mean_mag_ext < 20
GROUP BY healpix_, phot_g_mean_mag_, g_rp_
```
```
SELECT 
    floor(source_id / ((power(2, 35)) * (power(4, (12 - 3))))) AS healpix_,
    floor((phot_g_mean_mag_ext - 3) / 0.25) AS phot_g_mean_mag_,
    floor(((phot_g_mean_mag_ext - phot_rp_mean_mag_ext) - (-0.4)) / 0.1) AS g_rp_,
    COUNT(*) AS k
FROM (
    SELECT GAVO_NORMAL_RANDOM(phot_g_mean_mag, phot_g_mean_mag_error) + a0*A0_1_gaia_g as phot_g_mean_mag_ext,
           GAVO_NORMAL_RANDOM(phot_rp_mean_mag, phot_rp_mean_mag_error) + a0*A0_1_gaia_rp as phot_rp_mean_mag_ext,
           source_id,
           parallax
    FROM gedr3mock.main
    JOIN gedr3mock.parsec_props
    USING (index_parsec)) as mock
WHERE phot_g_mean_mag_ext - phot_rp_mean_mag_ext > -0.4
      AND phot_g_mean_mag_ext - phot_rp_mean_mag_ext < 2.2
      AND phot_g_mean_mag_ext > 3 
      AND phot_g_mean_mag_ext < 20 AND parallax > 10
GROUP BY healpix_, phot_g_mean_mag_, g_rp_
```
Below I load the results of these queries and join them.

In [1]:
from astropy.table import join, Table

mock_select = join(Table.read('query_results/gaiaedr3_mock_counts.fits', hdu=1),
                   Table.read('query_results/gaiaedr3_mock_counts.fits', hdu=2),
                   join_type='left', keys=['healpix_', 'phot_g_mean_mag_', 'g_rp_'])

mock_select = mock_select.filled(0)

mock_select['healpix_'] = mock_select['healpix_'].astype(int)
mock_select['phot_g_mean_mag_'] = mock_select['phot_g_mean_mag_'].astype(int)
mock_select['g_rp_'] = mock_select['g_rp_'].astype(int)

mock_select['n'][mock_select['k'] > mock_select['n']] = mock_select['k'][mock_select['k'] > mock_select['n']]

mock_select['k'].name = 'km'
mock_select['n'].name = 'nm'

# Query Gaia DR3

A similar query can be run on the [Gaia archive](https://gea.esac.esa.int/archive/) to get the observed number of stars in each bin within the Gaia DR3 catalog:
```
SELECT healpix_, phot_g_mean_mag_, g_rp_, COUNT(*) AS n
FROM (
	SELECT to_integer(GAIA_HEALPIX_INDEX(3, source_id)) AS healpix_,
	       to_integer(floor((phot_g_mean_mag - 3) / 0.25)) AS phot_g_mean_mag_,
	       to_integer(floor((g_rp - (-0.4)) / 0.1)) AS g_rp_
	FROM gaiadr3.gaia_source as gdr3
	WHERE phot_g_mean_mag > 3 AND phot_g_mean_mag < 20 AND
	      g_rp > -0.4 AND g_rp < 2.2)
AS subquery
GROUP BY healpix_, phot_g_mean_mag_, g_rp_
```
The result is then joined to the mock catalog results from the above

In [2]:
subSF = Table.read('query_results/gaiadr3_counts.csv')

In [3]:
subSF_mock = join(subSF,
                  mock_select,
                  join_type='left',
                  keys=['healpix_', 'phot_g_mean_mag_', 'g_rp_'])

subSF_mock = subSF_mock.filled(0)

Save the results to codebase. This is used as the default data to construct the selection function.

In [4]:
bin_str = "{'healpix': 3, 'phot_g_mean_mag': [3, 20, 0.25], 'g_rp': [-0.4, 2.2, 0.1]}"

with open('../src/snc_sf/sf_files/100pc_SF.csv', "w") as f:
    f.write(f"#{bin_str}\n")
    subSF_mock.write(f, format='csv')
